# Automatron Space

Conjunction triage, anomaly diagnosis support, and licensing paperwork.

## Contents

1. Constants and thresholds
2. Sample data
3. Conjunction tools
4. Anomaly tools
5. Licensing tools
6. Prompt addenda and workflows
7. Sector pack

Build the core module and put the generated package on the path. This cell is
for interactive use only and is dropped from the built module.

In [ ]:
import pathlib
import subprocess
import sys

ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
subprocess.run([sys.executable, "scripts/build_notebooks.py"], check=True, cwd=ROOT)
sys.path.insert(0, str(ROOT / "automatron_build"))

In [ ]:
from automatron_core import *  # noqa: F401,F403

## 1. Constants and thresholds

Values read from config/sectors.yaml and the rule files.

In [ ]:
import datetime as dt
import functools
import json
import math
import pathlib
import re
import time
from typing import Any

import httpx
import numpy as np
import yaml

SECTOR_ID = "space"
THRESHOLDS = sector_settings(SECTOR_ID)["thresholds"]

PC_RED = float(THRESHOLDS["pc_red"])
PC_YELLOW = float(THRESHOLDS["pc_yellow"])
HARD_BODY_RADIUS_M = float(THRESHOLDS["hard_body_radius_m"])
OD_AGE_WARN_DAYS = float(THRESHOLDS["od_age_warn_days"])
JUMP_ORDERS = float(THRESHOLDS["jump_orders"])

SAMPLE_DIR = get_settings().data_path / "samples" / SECTOR_ID
RULES_DIR = ROOT / "config" / "rules"

# Covariance scale factors for the dilution check: a low Pc that rises sharply
# when the covariance is scaled is an artefact of poor tracking, not safety.
DILUTION_SCALES = (0.1, 0.25, 0.5, 1.0, 2.0, 5.0, 10.0)
PC_GRID = 240
CELESTRAK_GP_URL = "https://celestrak.org/NORAD/elements/gp.php"
GP_CACHE_HOURS = 2


@functools.lru_cache(maxsize=2)
def load_fmea() -> dict[str, Any]:
    with (RULES_DIR / "space_fmea.yaml").open(encoding="utf-8") as handle:
        return yaml.safe_load(handle)


@functools.lru_cache(maxsize=2)
def load_licensing_rules() -> dict[str, Any]:
    with (RULES_DIR / "space_licensing.yaml").open(encoding="utf-8") as handle:
        return yaml.safe_load(handle)

## 2. Sample data

ensure_samples(): deterministic CDM series, telemetry, registry and mission profiles.

In [ ]:
CDM_TEMPLATE = """CCSDS_CDM_VERS = 1.0
CREATION_DATE = {created}
ORIGINATOR = SAMPLE_SSA_PROVIDER
MESSAGE_FOR = {primary_name}
MESSAGE_ID = {message_id}
COMMENT Synthetic conjunction data message written for this project. Not operational data.
TCA = {tca}
MISS_DISTANCE = {miss_m:.3f}
RELATIVE_SPEED = {rel_speed:.3f}
RELATIVE_POSITION_R = {rpos_r:.3f}
RELATIVE_POSITION_T = {rpos_t:.3f}
RELATIVE_POSITION_N = {rpos_n:.3f}
RELATIVE_VELOCITY_R = {rvel_r:.3f}
RELATIVE_VELOCITY_T = {rvel_t:.3f}
RELATIVE_VELOCITY_N = {rvel_n:.3f}
OBJECT = OBJECT1
OBJECT_DESIGNATOR = {primary_id}
OBJECT_NAME = {primary_name}
INTERNATIONAL_DESIGNATOR = {primary_intl}
OBJECT_TYPE = PAYLOAD
TIME_LASTOB_START = {od1_epoch}
TIME_LASTOB_END = {od1_epoch}
CR_R = {p_rr:.6e}
CT_R = {p_tr:.6e}
CT_T = {p_tt:.6e}
CN_R = {p_nr:.6e}
CN_T = {p_nt:.6e}
CN_N = {p_nn:.6e}
OBJECT = OBJECT2
OBJECT_DESIGNATOR = {secondary_id}
OBJECT_NAME = {secondary_name}
INTERNATIONAL_DESIGNATOR = {secondary_intl}
OBJECT_TYPE = ROCKET BODY
TIME_LASTOB_START = {od2_epoch}
TIME_LASTOB_END = {od2_epoch}
CR_R = {s_rr:.6e}
CT_R = {s_tr:.6e}
CT_T = {s_tt:.6e}
CN_R = {s_nr:.6e}
CN_T = {s_nt:.6e}
CN_N = {s_nn:.6e}
"""

TCA_EPOCH = dt.datetime(2026, 3, 3, 12, 0, 0, tzinfo=dt.UTC)


def miss_for_target_pc(target_pc: float, sigma_p: float, sigma_s: float) -> float:
    """Find the miss distance that puts a sample conjunction at a chosen Pc.

    Solved by bisection against the same integrator the tools use. Inverting the
    isotropic approximation instead would miss, because these samples carry an
    anisotropic covariance with a larger along-track term.
    """

    def pc_at(miss: float) -> float:
        text = _cdm_text(0, TCA_EPOCH, miss, sigma_p, sigma_s, 0.5, "PROBE")
        message = parse_cdm.invoke({"text": text})["messages"][0]
        return _pc_for_message(message)

    low, high = 0.0, 20000.0
    # Pc falls as the miss grows, so bisect on that monotonic relationship.
    for _ in range(60):
        middle = (low + high) / 2
        if pc_at(middle) > target_pc:
            low = middle
        else:
            high = middle
    return round((low + high) / 2, 3)


def _cdm_text(index, created, miss_m, sigma_p, sigma_s, od_age_days, message_id):
    """One CDM. The miss vector sits in the encounter plane, mostly along-track."""
    rel_speed = 14200.0
    # Split the miss between radial and cross-track so the projection is non-trivial.
    rpos_r, rpos_n = miss_m * 0.6, miss_m * 0.8
    covariance_p = {"p_rr": sigma_p**2, "p_tr": 0.0, "p_tt": (sigma_p * 3) ** 2,
                    "p_nr": 0.0, "p_nt": 0.0, "p_nn": sigma_p**2}
    covariance_s = {"s_rr": sigma_s**2, "s_tr": 0.0, "s_tt": (sigma_s * 3) ** 2,
                    "s_nr": 0.0, "s_nt": 0.0, "s_nn": sigma_s**2}
    od_epoch = (TCA_EPOCH - dt.timedelta(days=od_age_days)).strftime("%Y-%m-%dT%H:%M:%S.000")
    return CDM_TEMPLATE.format(
        created=created.strftime("%Y-%m-%dT%H:%M:%S.000"),
        message_id=message_id,
        tca=TCA_EPOCH.strftime("%Y-%m-%dT%H:%M:%S.000"),
        miss_m=miss_m, rel_speed=rel_speed,
        rpos_r=rpos_r, rpos_t=0.0, rpos_n=rpos_n,
        rvel_r=5.0, rvel_t=-rel_speed, rvel_n=30.0,
        primary_id=46001, primary_name="SAMPLESAT-1", primary_intl="2026-001A",
        secondary_id=27831, secondary_name="SAMPLE DEBRIS OBJ", secondary_intl="2003-014C",
        od1_epoch=(TCA_EPOCH - dt.timedelta(days=0.5)).strftime("%Y-%m-%dT%H:%M:%S.000"),
        od2_epoch=od_epoch,
        **covariance_p, **covariance_s,
    )


def _series(name, targets, sigma_p, sigma_s, od_ages, misses=None):
    """Build a CDM series whose Pc follows the given path.

    Pass explicit miss distances where the geometry matters more than the exact
    probability, as it does for the dilution scenario.
    """
    texts = []
    for index, (target, od_age) in enumerate(zip(targets, od_ages, strict=True)):
        created = TCA_EPOCH - dt.timedelta(hours=72 - index * 18)
        # Secondary uncertainty may vary across the series, as real tracking does.
        sigma_s_i = sigma_s[index] if isinstance(sigma_s, list) else sigma_s
        miss = misses[index] if misses else miss_for_target_pc(target, sigma_p, sigma_s_i)
        texts.append(
            _cdm_text(index, created, miss, sigma_p, sigma_s_i, od_age,
                      f"{name.upper()}-{index + 1:03d}")
        )
    return "\n".join(texts)


SAMPLE_GENERATOR_VERSION = 4


def ensure_samples(force: bool = False) -> None:
    """Write the bundled scenarios. Deterministic, so re-running changes nothing."""
    SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
    stamp = SAMPLE_DIR / ".generator_version"
    current = stamp.read_text(encoding="utf-8").strip() if stamp.is_file() else ""
    if not force and current == str(SAMPLE_GENERATOR_VERSION) and \
            (SAMPLE_DIR / "cdm_high_risk.cdm").is_file():
        return

    scenarios = {
        # Falls comfortably below the monitoring threshold.
        "cdm_low_risk": _series("cdm_low_risk", [8e-6, 2e-6, 4e-7, 6e-8],
                                90.0, 140.0, [0.4, 0.4, 0.3, 0.2]),
        # Climbs through the red threshold.
        "cdm_high_risk": _series("cdm_high_risk", [4e-6, 3e-5, 1.4e-4, 3.0e-4],
                                 80.0, 120.0, [0.5, 0.4, 0.4, 0.3]),
        # Jumps two orders on stale, inflated tracking: the dilution case.
        # Small miss against very large, stale covariance: the reported probability
        # sits past the peak, so shrinking the covariance raises it sharply.
        # Tracking degrades across the series: the covariance inflates and the
        # reported probability jumps, while the latest report sits past the peak.
        "cdm_jumpy_stale": _series("cdm_jumpy_stale", [0, 0, 0, 0],
                                   150.0, [260.0, 300.0, 2600.0, 3000.0],
                                   [1.0, 2.0, 5.0, 5.4],
                                   misses=[1500.0, 1450.0, 240.0, 200.0]),
    }
    for name, text in scenarios.items():
        (SAMPLE_DIR / f"{name}.cdm").write_text(text, encoding="utf-8")

    _write_telemetry()
    _write_spectrum_registry()
    _write_missions()
    stamp.write_text(str(SAMPLE_GENERATOR_VERSION), encoding="utf-8")


def _write_telemetry() -> None:
    """Two days at one-minute cadence for two fault scenarios."""
    import pandas as pd

    minutes = 2 * 24 * 60
    start = dt.datetime(2026, 2, 1, tzinfo=dt.UTC)
    stamps = [start + dt.timedelta(minutes=i) for i in range(minutes)]
    orbit_period = 95
    eclipse = np.array([1 if (i % orbit_period) > 60 else 0 for i in range(minutes)])
    rng = np.random.default_rng(20260201)

    # Reaction wheel fault: current and temperature climb, speed oscillates.
    ramp = np.linspace(0.0, 1.0, minutes)
    frame = pd.DataFrame({
        "time": [s.strftime("%Y-%m-%dT%H:%M:%SZ") for s in stamps],
        "battery_v": np.round(28.4 - 0.7 * eclipse + rng.normal(0, 0.02, minutes), 3),
        "bus_current_a": np.round(4.1 + 0.3 * eclipse + rng.normal(0, 0.03, minutes), 3),
        "panel_temp_c": np.round(18 - 26 * eclipse + rng.normal(0, 0.4, minutes), 2),
        "rw1_rpm": np.round(3200 + 140 * ramp * np.sin(np.arange(minutes) * 0.42)
                            + rng.normal(0, 6, minutes), 1),
        "rw1_current_a": np.round(0.42 + 0.36 * ramp + rng.normal(0, 0.006, minutes), 4),
        "rw1_temp_c": np.round(21 + 6.2 * ramp + rng.normal(0, 0.15, minutes), 2),
        "heater_on": np.where(eclipse > 0, 1, 0),
        "eclipse": eclipse,
    })
    frame.to_csv(SAMPLE_DIR / "telemetry_rw_fault.csv", index=False)

    # Battery scenario: a heater stays on through eclipse exit from hour 12.
    stuck = np.array([1 if i > 12 * 60 else 0 for i in range(minutes)])
    heater = np.where((eclipse > 0) | (stuck > 0), 1, 0)
    extra_load = 0.55 * stuck
    frame2 = pd.DataFrame({
        "time": frame["time"],
        "battery_v": np.round(28.4 - 0.7 * eclipse - 0.9 * stuck * eclipse
                              + rng.normal(0, 0.02, minutes), 3),
        "bus_current_a": np.round(4.1 + 0.3 * eclipse + extra_load
                                  + rng.normal(0, 0.03, minutes), 3),
        "panel_temp_c": np.round(18 - 26 * eclipse + rng.normal(0, 0.4, minutes), 2),
        "rw1_rpm": np.round(3200 + rng.normal(0, 6, minutes), 1),
        "rw1_current_a": np.round(0.42 + rng.normal(0, 0.006, minutes), 4),
        "rw1_temp_c": np.round(21 + rng.normal(0, 0.15, minutes), 2),
        "heater_on": heater,
        "eclipse": eclipse,
    })
    frame2.to_csv(SAMPLE_DIR / "telemetry_battery_eclipse.csv", index=False)


def _write_spectrum_registry() -> None:
    import pandas as pd

    rows = [
        ("HARBOUR-1", "Harbourline Communications", "Ku", 11450.0, 250.0, "downlink", 550, 53.0),
        ("HARBOUR-2", "Harbourline Communications", "Ku", 11700.0, 250.0, "downlink", 560, 53.0),
        ("NORTHSTAR-A", "Northstar Imaging", "X", 8150.0, 375.0, "downlink", 620, 97.8),
        ("NORTHSTAR-B", "Northstar Imaging", "Ku", 11300.0, 150.0, "downlink", 615, 97.6),
        ("MERIDIAN-3", "Meridian Geo Services", "Ka", 19700.0, 500.0, "downlink", 35786, 0.1),
        ("PELICAN-7", "Pelican Smallsat Co", "S", 2225.0, 4.0, "downlink", 500, 97.4),
        ("PELICAN-8", "Pelican Smallsat Co", "S", 2245.0, 4.0, "uplink", 500, 97.4),
        ("ORCA-1", "Orca Earth Systems", "X", 8025.0, 300.0, "downlink", 700, 98.2),
    ]
    pd.DataFrame(rows, columns=[
        "satellite", "operator", "band", "center_mhz", "bandwidth_mhz",
        "direction", "altitude_km", "inclination_deg",
    ]).to_csv(SAMPLE_DIR / "spectrum_registry.csv", index=False)


def _write_missions() -> None:
    missions = {
        "mission_eo_leo": {
            "mission_name": "Northlight-1",
            "operator": "Northlight Imaging (fictional)",
            "activity_type": "earth_observation",
            "orbit": {"altitude_km": 610, "inclination_deg": 97.8, "type": "sun_synchronous"},
            "frequencies": [
                {"band": "X", "center_mhz": 8200.0, "bandwidth_mhz": 300.0,
                 "direction": "downlink", "eirp_dbw": 12.5},
                {"band": "S", "center_mhz": 2230.0, "bandwidth_mhz": 4.0,
                 "direction": "uplink", "eirp_dbw": 3.0},
            ],
            "launch_site": "Vandenberg SFB",
            "launch_date": "2027-04-15",
            "jurisdictions": ["US"],
        },
        "mission_servicing": {
            "mission_name": "Tender-1",
            "operator": "Orbital Tender Works (fictional)",
            "activity_type": "in_space_servicing",
            "orbit": {"altitude_km": 35786, "inclination_deg": 0.1, "type": "geostationary"},
            "frequencies": [
                {"band": "Ka", "center_mhz": 19750.0, "bandwidth_mhz": 500.0,
                 "direction": "downlink", "eirp_dbw": 20.0},
            ],
            "launch_site": "Cape Canaveral SFS",
            "launch_date": "2028-01-20",
            "jurisdictions": ["US"],
        },
    }
    for name, payload in missions.items():
        (SAMPLE_DIR / f"{name}.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")

## 3. Conjunction tools

CDM parsing, the encounter-plane probability of collision,\ntrend across a series, covariance quality with the dilution check, and\nclassification against operator thresholds.

In [ ]:
class ParseCdmArgs(BaseModel):
    text: str = Field(default="", description="One or more CCSDS CDMs in key-value notation.")
    sample_name: str = Field(default="", description="Bundled CDM series to read instead.")


class ComputePcArgs(BaseModel):
    rel_pos_m: list[float] = Field(description="Relative position in RTN metres.")
    rel_vel_mps: list[float] = Field(description="Relative velocity in RTN metres per second.")
    cov1_m2: list[list[float]] = Field(description="Primary 3x3 position covariance, m^2.")
    cov2_m2: list[list[float]] = Field(description="Secondary 3x3 position covariance, m^2.")
    hbr_m: float = Field(default=HARD_BODY_RADIUS_M, description="Combined hard-body radius.")


class PcTrendArgs(BaseModel):
    cdm_series: list[dict[str, Any]] = Field(default_factory=list,
                                             description="Parsed CDMs, oldest first.")
    sample_name: str = Field(default="", description="Bundled CDM series to read instead.")


class CovarianceQualityArgs(BaseModel):
    cdm: dict[str, Any] = Field(default_factory=dict, description="One parsed CDM.")
    sample_name: str = Field(default="", description="Bundled series; the latest message is used.")


class ClassifyArgs(BaseModel):
    pc: float | None = None
    trend: dict[str, Any] = Field(default_factory=dict)
    quality: dict[str, Any] = Field(default_factory=dict)
    policy: dict[str, float] = Field(default_factory=dict)
    sample_name: str = Field(default="", description="Compute everything from a bundled series.")


class FetchGpArgs(BaseModel):
    norad_id: int = Field(description="NORAD catalogue number.")


def _sample_cdm_text(sample_name: str) -> str:
    ensure_samples()
    stem = sample_name.removesuffix(".cdm") or "cdm_high_risk"
    path = SAMPLE_DIR / f"{stem}.cdm"
    if not path.is_file():
        raise FileNotFoundError(f"no bundled CDM series named '{stem}'")
    return path.read_text(encoding="utf-8")


def _series_from(cdm_series, sample_name):
    """Take parsed messages as given, or parse a bundled series.

    Tools accept a sample name so each one is usable on its own, which is also what
    lets demo mode script a step without threading state between tool calls.
    """
    if cdm_series:
        return list(cdm_series)
    return parse_cdm.invoke({"sample_name": sample_name or "cdm_high_risk"})["messages"]


def _cdm_blocks(text: str) -> list[dict[str, str]]:
    """Split a KVN CDM into its header and the two object blocks."""
    header: dict[str, str] = {}
    objects: list[dict[str, str]] = []
    current = header
    for raw in text.splitlines():
        line = raw.strip()
        if not line or line.startswith("COMMENT"):
            continue
        if "=" not in line:
            continue
        key, _, value = line.partition("=")
        key, value = key.strip().upper(), value.strip()
        if key == "OBJECT" and value.upper().startswith("OBJECT"):
            current = {}
            objects.append(current)
            continue
        current[key] = value
    return [header, *objects]


def _covariance_from(block: dict[str, str]) -> list[list[float]]:
    """Rebuild the symmetric 3x3 position covariance from the CDM's lower triangle."""

    def value(key: str) -> float:
        try:
            return float(block.get(key, 0.0))
        except ValueError:
            return 0.0

    rr, tr, tt = value("CR_R"), value("CT_R"), value("CT_T")
    nr, nt, nn = value("CN_R"), value("CN_T"), value("CN_N")
    return [[rr, tr, nr], [tr, tt, nt], [nr, nt, nn]]


def _parse_time(value: str) -> dt.datetime | None:
    """Accept CDM timestamps and the ISO strings this module emits when reparsing.

    The parser writes offset-aware ISO strings, so anything that reads its output
    back has to handle the offset as well as the bare CDM forms.
    """
    if not value:
        return None
    try:
        parsed = dt.datetime.fromisoformat(value.replace("Z", "+00:00"))
        return parsed if parsed.tzinfo else parsed.replace(tzinfo=dt.UTC)
    except ValueError:
        pass
    for pattern in ("%Y-%m-%dT%H:%M:%S.%f", "%Y-%m-%dT%H:%M:%S"):
        try:
            return dt.datetime.strptime(value, pattern).replace(tzinfo=dt.UTC)
        except ValueError:
            continue
    return None


@tool(args_schema=ParseCdmArgs)
def parse_cdm(text: str = "", sample_name: str = "") -> dict[str, Any]:
    """Parse one or more CCSDS conjunction data messages into structured records.

    Returns the messages ordered by creation time, each with the time of closest
    approach, miss distance, relative position and velocity in the RTN frame, both
    position covariances, and the orbit determination epochs.
    """
    if not text.strip():
        try:
            text = _sample_cdm_text(sample_name)
        except FileNotFoundError as exc:
            return {"error": str(exc), "messages": [], "count": 0, "tool_version": 1}

    chunks = re.split(r"(?=CCSDS_CDM_VERS)", text.strip())
    messages: list[dict[str, Any]] = []
    problems: list[str] = []

    for chunk in [c for c in chunks if c.strip()]:
        header, *objects = _cdm_blocks(chunk)
        if len(objects) < 2:
            problems.append("a message did not contain two object blocks")
            continue
        missing = [k for k in ("TCA", "MISS_DISTANCE") if k not in header]
        if missing:
            problems.append(f"a message is missing required fields: {missing}")
            continue

        tca = _parse_time(header.get("TCA", ""))
        created = _parse_time(header.get("CREATION_DATE", ""))
        primary, secondary = objects[0], objects[1]
        messages.append({
            "message_id": header.get("MESSAGE_ID", ""),
            "created_utc": created.isoformat() if created else None,
            "tca_utc": tca.isoformat() if tca else None,
            "miss_distance_m": float(header.get("MISS_DISTANCE", "nan")),
            "relative_speed_mps": float(header.get("RELATIVE_SPEED", "nan")),
            "rel_pos_rtn_m": [float(header.get(f"RELATIVE_POSITION_{a}", 0.0)) for a in "RTN"],
            "rel_vel_rtn_mps": [float(header.get(f"RELATIVE_VELOCITY_{a}", 0.0)) for a in "RTN"],
            "primary": {
                "designator": primary.get("OBJECT_DESIGNATOR", ""),
                "name": primary.get("OBJECT_NAME", ""),
                "covariance_m2": _covariance_from(primary),
                "od_epoch_utc": (_parse_time(primary.get("TIME_LASTOB_END", "")) or tca
                                 or TCA_EPOCH).isoformat(),
            },
            "secondary": {
                "designator": secondary.get("OBJECT_DESIGNATOR", ""),
                "name": secondary.get("OBJECT_NAME", ""),
                "covariance_m2": _covariance_from(secondary),
                "od_epoch_utc": (_parse_time(secondary.get("TIME_LASTOB_END", "")) or tca
                                 or TCA_EPOCH).isoformat(),
            },
        })

    messages.sort(key=lambda m: m["created_utc"] or "")
    return {"messages": messages, "count": len(messages), "problems": problems,
            "tool_version": 1}


def _encounter_plane(rel_pos, rel_vel, cov1, cov2):
    """Project the combined covariance and the miss vector into the encounter plane."""
    r = np.asarray(rel_pos, dtype=float)
    v = np.asarray(rel_vel, dtype=float)
    combined = np.asarray(cov1, dtype=float) + np.asarray(cov2, dtype=float)

    speed = np.linalg.norm(v)
    if speed <= 0:
        raise ValueError("relative velocity must be non-zero")
    e_y = v / speed

    projected = r - np.dot(r, e_y) * e_y
    norm = np.linalg.norm(projected)
    if norm > 1e-9:
        e_x = projected / norm
    else:
        fallback = np.eye(3)[int(np.argmin(np.abs(e_y)))]
        e_x = fallback - np.dot(fallback, e_y) * e_y
        e_x /= np.linalg.norm(e_x)
    e_z = np.cross(e_x, e_y)

    basis = np.vstack([e_x, e_z])
    return basis @ combined @ basis.T, basis @ r


@tool(args_schema=ComputePcArgs)
def compute_pc_2d(rel_pos_m: list[float], rel_vel_mps: list[float],
                  cov1_m2: list[list[float]], cov2_m2: list[list[float]],
                  hbr_m: float = HARD_BODY_RADIUS_M) -> dict[str, Any]:
    """Probability of collision by the two-dimensional encounter-plane method.

    The combined covariance is projected into the plane perpendicular to the
    relative velocity and the resulting Gaussian is integrated numerically over a
    circle of the combined hard-body radius. Assumes a short, linear encounter with
    Gaussian position errors.
    """
    try:
        cov_2d, miss_2d = _encounter_plane(rel_pos_m, rel_vel_mps, cov1_m2, cov2_m2)
    except ValueError as exc:
        return {"error": str(exc), "tool_version": 1}

    determinant = float(np.linalg.det(cov_2d))
    if determinant <= 0:
        return {"error": "projected covariance is not positive definite",
                "determinant": determinant, "tool_version": 1}

    inverse = np.linalg.inv(cov_2d)
    radii = (np.arange(PC_GRID) + 0.5) / PC_GRID * hbr_m
    angles = (np.arange(PC_GRID) + 0.5) / PC_GRID * 2 * np.pi
    rr, aa = np.meshgrid(radii, angles, indexing="ij")
    dx = miss_2d[0] + rr * np.cos(aa)
    dy = miss_2d[1] + rr * np.sin(aa)

    quad = inverse[0, 0] * dx * dx + 2 * inverse[0, 1] * dx * dy + inverse[1, 1] * dy * dy
    density = np.exp(-0.5 * quad) / (2 * np.pi * math.sqrt(determinant))
    weight = rr * (hbr_m / PC_GRID) * (2 * np.pi / PC_GRID)
    pc = float(np.sum(density * weight))

    eigenvalues = np.linalg.eigvalsh(cov_2d)
    mahalanobis = float(math.sqrt(max(miss_2d @ inverse @ miss_2d, 0.0)))
    return {
        "pc": pc,
        "pc_scientific": f"{pc:.2e}",
        "sigma_x_m": float(math.sqrt(max(eigenvalues[-1], 0.0))),
        "sigma_z_m": float(math.sqrt(max(eigenvalues[0], 0.0))),
        "projected_miss_m": float(np.linalg.norm(miss_2d)),
        "mahalanobis": mahalanobis,
        "hbr_m": hbr_m,
        "method": "2D encounter plane, numerically integrated",
        "assumptions": [
            "short encounter with a linear relative trajectory",
            "Gaussian position errors",
            "covariances are uncorrelated between the two objects",
        ],
        "tool_version": 1,
    }


def _pc_for_message(message: dict[str, Any], hbr_m: float = HARD_BODY_RADIUS_M,
                    scale: float = 1.0) -> float:
    cov1 = (np.asarray(message["primary"]["covariance_m2"], dtype=float) * scale).tolist()
    cov2 = (np.asarray(message["secondary"]["covariance_m2"], dtype=float) * scale).tolist()
    result = compute_pc_2d.invoke({
        "rel_pos_m": message["rel_pos_rtn_m"],
        "rel_vel_mps": message["rel_vel_rtn_mps"],
        "cov1_m2": cov1, "cov2_m2": cov2, "hbr_m": hbr_m,
    })
    return float(result.get("pc", float("nan")))


@tool(args_schema=PcTrendArgs)
def pc_trend(cdm_series: list[dict[str, Any]] | None = None,
             sample_name: str = "") -> dict[str, Any]:
    """Probability of collision across a CDM series, with jumps and threshold crossings.

    A jump of an order of magnitude or more between consecutive messages usually
    reflects new tracking data rather than a changed physical situation.
    """
    cdm_series = _series_from(cdm_series, sample_name)
    if not cdm_series:
        return {"error": "no messages supplied", "tool_version": 1}

    points = []
    for message in cdm_series:
        pc = _pc_for_message(message)
        points.append({
            "message_id": message.get("message_id", ""),
            "created_utc": message.get("created_utc"),
            "pc": pc,
            "pc_scientific": f"{pc:.2e}",
            "miss_distance_m": message.get("miss_distance_m"),
        })

    jumps = []
    for earlier, later in zip(points, points[1:], strict=False):
        if earlier["pc"] > 0 and later["pc"] > 0:
            orders = math.log10(later["pc"] / earlier["pc"])
            if abs(orders) >= JUMP_ORDERS:
                jumps.append({
                    "from_message": earlier["message_id"], "to_message": later["message_id"],
                    "orders_of_magnitude": round(orders, 2),
                    "direction": "increase" if orders > 0 else "decrease",
                })

    crossings = []
    for earlier, later in zip(points, points[1:], strict=False):
        for name, level in (("red", PC_RED), ("yellow", PC_YELLOW)):
            if earlier["pc"] < level <= later["pc"]:
                crossings.append({"threshold": name, "level": f"{level:.0e}", "direction": "up"})
            elif later["pc"] < level <= earlier["pc"]:
                crossings.append({"threshold": name, "level": f"{level:.0e}", "direction": "down"})

    latest, first = points[-1]["pc"], points[0]["pc"]
    direction = "stable"
    if latest > first * 2:
        direction = "rising"
    elif latest < first / 2:
        direction = "falling"

    return {"points": points, "latest_pc": latest, "latest_pc_scientific": f"{latest:.2e}",
            "direction": direction, "jumps": jumps, "threshold_crossings": crossings,
            "message_count": len(points), "tool_version": 1}


@tool(args_schema=CovarianceQualityArgs)
def covariance_quality(cdm: dict[str, Any] | None = None,
                       sample_name: str = "") -> dict[str, Any]:
    """Check whether a probability of collision can be trusted.

    Covers positive-definiteness, how old the tracking is, unusually large
    uncertainties, and probability dilution: a low Pc that rises sharply when the
    covariance is scaled reflects poor tracking rather than a comfortable miss.
    """
    if not cdm:
        series = _series_from(None, sample_name)
        if not series:
            return {"error": "no message supplied", "tool_version": 1}
        cdm = series[-1]

    issues: list[str] = []
    tca = _parse_time(cdm.get("tca_utc", "") or "") or TCA_EPOCH
    details = {}

    for role in ("primary", "secondary"):
        block = cdm.get(role, {})
        covariance = np.asarray(block.get("covariance_m2", np.zeros((3, 3))), dtype=float)
        eigenvalues = np.linalg.eigvalsh(covariance)
        positive_definite = bool(np.all(eigenvalues > 0))
        if not positive_definite:
            issues.append(f"{role} covariance is not positive definite")

        sigmas = [float(math.sqrt(max(covariance[i][i], 0.0))) for i in range(3)]
        if max(sigmas) > 2000:
            issues.append(f"{role} position uncertainty is very large "
                          f"({max(sigmas):.0f} m along one axis)")

        od_epoch = _parse_time(block.get("od_epoch_utc", "") or "") or tca
        age_days = (tca - od_epoch).total_seconds() / 86400.0
        if age_days > OD_AGE_WARN_DAYS:
            issues.append(f"{role} tracking is {age_days:.1f} days old at the time of "
                          f"closest approach")
        details[role] = {
            "positive_definite": positive_definite,
            "sigma_rtn_m": [round(s, 1) for s in sigmas],
            "od_age_days": round(age_days, 2),
        }

    # Dilution: scale both covariances together and look for a higher reachable Pc.
    current = _pc_for_message(cdm)
    scaled = {}
    for scale in DILUTION_SCALES:
        scaled[str(scale)] = _pc_for_message(cdm, scale=scale)
    best_scale = max(scaled, key=lambda k: scaled[k])
    max_pc = scaled[best_scale]
    ratio = (max_pc / current) if current > 0 else float("inf")

    # Pc rises with uncertainty to a peak and falls away beyond it. Sitting past
    # that peak is dilution: the low number comes from poor tracking. Sitting
    # before it means better tracking would raise Pc, which is a different warning.
    diluted = float(best_scale) < 1.0 and ratio >= 3
    if diluted:
        issues.append(
            f"probability dilution: shrinking the covariance to {best_scale} of its reported "
            f"size reaches {max_pc:.2e}, {ratio:.0f} times the reported {current:.2e}"
        )
    elif float(best_scale) > 1.0 and ratio >= 10:
        issues.append(
            f"the reported probability is sensitive to tracking quality: inflating the "
            f"covariance to {best_scale} times reaches {max_pc:.2e}"
        )

    return {
        "current_pc": current, "current_pc_scientific": f"{current:.2e}",
        "max_pc_over_scales": max_pc, "max_pc_scientific": f"{max_pc:.2e}",
        "dilution_ratio": round(ratio, 1) if math.isfinite(ratio) else None,
        "peak_scale": float(best_scale),
        "dilution_warning": diluted,
        "pc_by_scale": {k: f"{v:.2e}" for k, v in scaled.items()},
        "per_object": details, "issues": issues, "tool_version": 1,
    }


@tool(args_schema=ClassifyArgs)
def classify_conjunction(pc: float | None = None, trend: dict[str, Any] | None = None,
                         quality: dict[str, Any] | None = None,
                         policy: dict[str, float] | None = None,
                         sample_name: str = "") -> dict[str, Any]:
    """Assign RED, YELLOW, GREEN or INSUFFICIENT_DATA from thresholds and data quality.

    Deterministic. Thresholds are operator policy, not physical constants.
    """
    if pc is None:
        series = _series_from(None, sample_name)
        trend = trend or pc_trend.invoke({"cdm_series": series})
        quality = quality or covariance_quality.invoke({"cdm": series[-1]})
        pc = float(trend.get("latest_pc", float("nan")))

    trend = trend or {}
    quality = quality or {}
    policy = policy or {}
    red = float(policy.get("pc_red", PC_RED))
    yellow = float(policy.get("pc_yellow", PC_YELLOW))

    reasons: list[str] = []
    blocking = list(quality.get("issues", []))

    if quality.get("dilution_warning"):
        level = "INSUFFICIENT_DATA"
        reasons.append("the reported probability is diluted by poor tracking")
    elif not math.isfinite(pc):
        level = "INSUFFICIENT_DATA"
        reasons.append("probability of collision could not be computed")
    elif pc >= red:
        level = "RED"
        reasons.append(f"Pc {pc:.2e} is at or above the red threshold {red:.0e}")
    elif pc >= yellow:
        level = "YELLOW"
        reasons.append(f"Pc {pc:.2e} is between the yellow {yellow:.0e} and red {red:.0e} "
                       "thresholds")
    else:
        level = "GREEN"
        reasons.append(f"Pc {pc:.2e} is below the yellow threshold {yellow:.0e}")

    if trend.get("jumps"):
        reasons.append(f"{len(trend['jumps'])} order-of-magnitude jump(s) across the series")
    if trend.get("direction") == "rising" and level == "YELLOW":
        reasons.append("the trend is rising, so the next message may cross the red threshold")

    return {"level": level, "reasons": reasons, "data_quality_issues": blocking,
            "thresholds_used": {"pc_red": f"{red:.0e}", "pc_yellow": f"{yellow:.0e}"},
            "tool_version": 1}


@tool(args_schema=FetchGpArgs)
def fetch_gp_elements(norad_id: int) -> dict[str, Any]:
    """Fetch current orbital elements for a catalogued object from CelesTrak.

    Cached on disk for at least two hours. Returns a clear error rather than raising
    when the service cannot be reached.
    """
    cache_dir = get_settings().runtime_path / "cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_file = cache_dir / f"gp_{norad_id}.json"

    if cache_file.is_file():
        age_hours = (time.time() - cache_file.stat().st_mtime) / 3600
        if age_hours < GP_CACHE_HOURS:
            return {**json.loads(cache_file.read_text(encoding="utf-8")), "cached": True,
                    "tool_version": 1}

    try:
        response = httpx.get(
            CELESTRAK_GP_URL,
            params={"CATNR": norad_id, "FORMAT": "json"},
            timeout=15,
            headers={"User-Agent": "Automatron/1.0 (+contact: arnavhpd@gmail.com)"},
        )
        response.raise_for_status()
        payload = response.json()
    except Exception as exc:
        return {"error": f"could not reach the catalogue: {redact(str(exc))[:160]}",
                "norad_id": norad_id, "tool_version": 1}

    if not payload:
        return {"error": f"no element set returned for {norad_id}", "tool_version": 1}
    record = {"norad_id": norad_id, "elements": payload[0],
              "source": "CelesTrak GP", "retrieved_utc": utcnow_iso()}
    cache_file.write_text(json.dumps(record), encoding="utf-8")
    return {**record, "cached": False, "tool_version": 1}

## 4. Anomaly tools

Telemetry loading, anomaly detection, lagged correlation,\ntimeline assembly, and failure-mode ranking.

In [ ]:
class LoadTelemetryArgs(BaseModel):
    sample_name: str = Field(default="", description="Bundled sample, e.g. telemetry_rw_fault.")
    file_path: str = Field(default="", description="Path to an uploaded CSV instead.")


class DetectAnomaliesArgs(BaseModel):
    dataset_id: str = Field(default="", description="From load_telemetry; the latest if empty.")
    channels: list[str] = Field(default_factory=list, description="Empty means every channel.")
    window: int = Field(default=60, description="Rolling window in samples.")


class CorrelateArgs(BaseModel):
    dataset_id: str = Field(default="", description="From load_telemetry; the latest if empty.")
    target: str = Field(description="Channel to explain.")
    max_lag: int = Field(default=120, description="Largest lag in samples to test.")


class TimelineArgs(BaseModel):
    dataset_id: str = Field(default="")
    intervals: list[dict[str, Any]] = Field(default_factory=list)


class RankHypothesesArgs(BaseModel):
    symptoms: list[str] = Field(description="Observed symptom keys.")


# Loaded frames stay in the process so several tools can share one parse.
_TELEMETRY_CACHE: dict[str, Any] = {}


def _frame_for(dataset_id: str):
    """The named dataset, or the one loaded most recently.

    Falling back to the latest lets a scripted demo step call these tools without
    knowing an id that only exists once the previous step has run.
    """
    if dataset_id and dataset_id in _TELEMETRY_CACHE:
        return _TELEMETRY_CACHE[dataset_id]
    if not dataset_id and _TELEMETRY_CACHE:
        return next(reversed(_TELEMETRY_CACHE.values()))
    return None

# Maps what the detectors see onto the vocabulary the failure table uses.
SYMPTOM_RULES = {
    "rw1_current_a": {"up": "rw_current_rise", "oscillation": "rw_speed_oscillation"},
    "rw1_temp_c": {"up": "rw_temp_rise"},
    "rw1_rpm": {"oscillation": "rw_speed_oscillation"},
    "battery_v": {"down": "battery_voltage_sag"},
    "bus_current_a": {"up": "bus_current_rise"},
    "panel_temp_c": {"up": "panel_temp_rise", "down": "panel_temp_low"},
}


def _load_frame(sample_name: str, file_path: str):
    import pandas as pd

    if file_path:
        path = pathlib.Path(file_path)
    else:
        ensure_samples()
        stem = sample_name.removesuffix(".csv") or "telemetry_rw_fault"
        path = SAMPLE_DIR / f"{stem}.csv"
    if not path.is_file():
        raise FileNotFoundError(f"no telemetry at {path.name}")
    return pd.read_csv(path), path


@tool(args_schema=LoadTelemetryArgs)
def load_telemetry(sample_name: str = "", file_path: str = "") -> dict[str, Any]:
    """Load a telemetry CSV and describe its channels.

    Returns a dataset id the other telemetry tools take, so the file is parsed once.
    """
    try:
        frame, path = _load_frame(sample_name, file_path)
    except (FileNotFoundError, ValueError) as exc:
        return {"error": str(exc), "tool_version": 1}

    dataset_id = f"tlm_{abs(hash(str(path))) % 10**8:08d}"
    _TELEMETRY_CACHE[dataset_id] = frame

    channels = {}
    for column in frame.columns:
        if column == "time":
            continue
        series = frame[column]
        channels[column] = {
            "min": round(float(series.min()), 4),
            "max": round(float(series.max()), 4),
            "mean": round(float(series.mean()), 4),
            "unit": _unit_for(column),
        }
    return {
        "dataset_id": dataset_id, "source": path.name, "rows": int(len(frame)),
        "start_utc": str(frame["time"].iloc[0]), "end_utc": str(frame["time"].iloc[-1]),
        "channels": channels, "tool_version": 1,
    }


def _unit_for(column: str) -> str:
    if column.endswith("_v"):
        return "V"
    if column.endswith("_a"):
        return "A"
    if column.endswith("_c"):
        return "degC"
    if column.endswith("_rpm"):
        return "rpm"
    return "flag"


@tool(args_schema=DetectAnomaliesArgs)
def detect_anomalies(dataset_id: str = "", channels: list[str] | None = None,
                     window: int = 60) -> dict[str, Any]:
    """Flag unusual intervals per channel using a robust rolling z-score and an outlier model.

    The z-score uses the median and median absolute deviation, so a sustained drift
    does not hide behind its own mean.
    """
    frame = _frame_for(dataset_id)
    if frame is None:
        return {"error": f"unknown dataset '{dataset_id}'; call load_telemetry first",
                "tool_version": 1}

    import pandas as pd
    from sklearn.ensemble import IsolationForest

    targets = [c for c in (channels or frame.columns) if c != "time" and c in frame.columns]
    findings = []
    for column in targets:
        series = pd.to_numeric(frame[column], errors="coerce").astype(float)
        if series.nunique() <= 1:
            continue
        median = series.rolling(window, min_periods=window // 2).median()
        deviation = (series - median).abs().rolling(window, min_periods=window // 2).median()
        scale = deviation.replace(0, np.nan) * 1.4826
        z = ((series - median) / scale).abs().fillna(0.0)
        flagged = z > 4.0

        forest = IsolationForest(random_state=0, contamination=0.02)
        outliers = forest.fit_predict(series.to_numpy().reshape(-1, 1)) == -1

        combined = flagged.to_numpy() | outliers
        share = float(combined.mean())
        first, last = None, None
        if combined.any():
            indices = np.flatnonzero(combined)
            first, last = int(indices[0]), int(indices[-1])

        start, end = series.iloc[: len(series) // 4], series.iloc[-len(series) // 4 :]
        change = float(end.mean() - start.mean())
        spread = float(series.std())
        direction = "stable"
        if spread > 0 and abs(change) > spread:
            direction = "up" if change > 0 else "down"

        # Oscillation: variance well above the residual of a smoothed version.
        smoothed = series.rolling(15, min_periods=5).mean()
        residual = float((series - smoothed).abs().mean())
        oscillating = bool(spread > 0 and residual > 0.35 * spread)

        findings.append({
            "channel": column,
            "flagged_fraction": round(share, 4),
            "first_flagged_index": first,
            "last_flagged_index": last,
            "direction": direction,
            "change": round(change, 4),
            "oscillating": oscillating,
            "unit": _unit_for(column),
        })

    symptoms = sorted({
        SYMPTOM_RULES.get(f["channel"], {}).get(key)
        for f in findings
        for key in ([f["direction"]] + (["oscillation"] if f["oscillating"] else []))
        if SYMPTOM_RULES.get(f["channel"], {}).get(key)
    })
    return {"findings": sorted(findings, key=lambda f: -f["flagged_fraction"]),
            "symptoms": symptoms, "window": window, "tool_version": 1}


@tool(args_schema=CorrelateArgs)
def correlate_channels(target: str, dataset_id: str = "", max_lag: int = 120) -> dict[str, Any]:
    """Rank other channels by lagged correlation with a target channel.

    The sign of the lag distinguishes a cause from a consequence: a channel that
    leads the target is a candidate driver, one that follows it is an effect.
    """
    frame = _frame_for(dataset_id)
    if frame is None:
        return {"error": f"unknown dataset '{dataset_id}'", "tool_version": 1}
    if target not in frame.columns:
        return {"error": f"no channel '{target}'", "tool_version": 1}

    import pandas as pd

    base = pd.to_numeric(frame[target], errors="coerce").astype(float)
    results = []
    for column in frame.columns:
        if column in ("time", target):
            continue
        other = pd.to_numeric(frame[column], errors="coerce").astype(float)
        if other.nunique() <= 1:
            continue
        best = {"lag": 0, "correlation": 0.0}
        for lag in range(-max_lag, max_lag + 1, 5):
            shifted = other.shift(lag)
            paired = pd.concat([base, shifted], axis=1).dropna()
            if len(paired) < 30:
                continue
            value = float(paired.iloc[:, 0].corr(paired.iloc[:, 1]))
            if math.isfinite(value) and abs(value) > abs(best["correlation"]):
                best = {"lag": lag, "correlation": round(value, 4)}
        if abs(best["correlation"]) > 0.3:
            results.append({
                "channel": column, "lag_samples": best["lag"],
                "correlation": best["correlation"],
                "relationship": "leads the target" if best["lag"] > 0 else
                                ("follows the target" if best["lag"] < 0 else "in step"),
            })

    results.sort(key=lambda r: -abs(r["correlation"]))
    return {"target": target, "top": results[:5], "tool_version": 1}


@tool(args_schema=TimelineArgs)
def build_timeline(dataset_id: str = "", intervals: list[dict[str, Any]] | None = None
                   ) -> dict[str, Any]:
    """Merge flagged intervals and discrete state changes into one ordered timeline."""
    frame = _frame_for(dataset_id)
    if frame is None:
        return {"error": f"unknown dataset '{dataset_id}'", "tool_version": 1}

    events = []
    for column in ("heater_on", "eclipse"):
        if column not in frame.columns:
            continue
        values = frame[column].astype(int).to_numpy()
        changes = np.flatnonzero(np.diff(values)) + 1
        for index in changes[:20]:
            events.append({
                "index": int(index), "time": str(frame["time"].iloc[int(index)]),
                "event": f"{column} -> {int(values[int(index)])}", "kind": "state_change",
            })

    for interval in intervals or []:
        index = int(interval.get("first_flagged_index") or 0)
        if 0 <= index < len(frame):
            events.append({
                "index": index, "time": str(frame["time"].iloc[index]),
                "event": f"{interval.get('channel', 'channel')} first flagged", "kind": "anomaly",
            })

    events.sort(key=lambda e: e["index"])
    return {"events": events[:40], "event_count": len(events), "tool_version": 1}


@tool(args_schema=RankHypothesesArgs)
def rank_hypotheses(symptoms: list[str]) -> dict[str, Any]:
    """Score failure modes against observed symptoms using the sector's failure table.

    Deterministic scoring, not a diagnosis: it reports which recorded failure modes
    the evidence matches and which of their symptoms were not observed.
    """
    table = load_fmea()["failure_modes"]
    observed = set(symptoms)
    ranked = []
    for mode in table:
        expected = set(mode["symptoms"])
        matched = sorted(observed & expected)
        if not matched:
            continue
        ranked.append({
            "id": mode["id"], "name": mode["name"], "subsystem": mode["subsystem"],
            "score": round(len(matched) / len(expected), 3),
            "matched_symptoms": matched,
            "unmatched_symptoms": sorted(expected - observed),
            "diagnostics": mode["diagnostics"],
            "safety_note": mode["safety_note"],
        })
    ranked.sort(key=lambda m: (-m["score"], m["id"]))
    return {"ranked": ranked[:5], "observed_symptoms": sorted(observed),
            "considered": len(table), "tool_version": 1}

## 5. Licensing tools

Profile validation, requirement checklists, spectrum overlap,\ndraft coordination letters, and indicative timelines.

In [ ]:
class MissionProfileArgs(BaseModel):
    profile: dict[str, Any] = Field(default_factory=dict, description="Mission profile JSON.")
    sample_name: str = Field(default="", description="Bundled mission profile to read instead.")


class ChecklistArgs(BaseModel):
    activity_type: str
    jurisdictions: list[str] = Field(default_factory=lambda: ["US"])


class AffectedOperatorsArgs(BaseModel):
    frequencies: list[dict[str, Any]]
    orbit: dict[str, Any] = Field(default_factory=dict)


class CoordinationLetterArgs(BaseModel):
    party: str
    overlaps: list[dict[str, Any]] = Field(default_factory=list)
    mission: dict[str, Any] = Field(default_factory=dict)


class TimelineEstimateArgs(BaseModel):
    checklist: dict[str, Any]


ACTIVITY_TYPES = ("earth_observation", "communications", "in_space_servicing",
                  "debris_removal", "in_space_manufacturing", "resource_extraction",
                  "launch", "reentry")


@tool(args_schema=MissionProfileArgs)
def validate_mission_profile(profile: dict[str, Any] | None = None,
                             sample_name: str = "") -> dict[str, Any]:
    """Check a mission profile for missing fields and values outside plausible ranges."""
    if not profile:
        ensure_samples()
        stem = (sample_name or "mission_eo_leo").removesuffix(".json")
        path = SAMPLE_DIR / f"{stem}.json"
        if not path.is_file():
            return {"error": f"no bundled mission named '{stem}'", "tool_version": 1}
        profile = json.loads(path.read_text(encoding="utf-8"))

    missing: list[str] = []
    warnings: list[str] = []

    for field in ("mission_name", "operator", "activity_type", "orbit", "frequencies"):
        if not profile.get(field):
            missing.append(field)

    activity = profile.get("activity_type", "")
    if activity and activity not in ACTIVITY_TYPES:
        warnings.append(f"activity_type '{activity}' is not one of {list(ACTIVITY_TYPES)}")

    orbit = profile.get("orbit") or {}
    altitude = orbit.get("altitude_km")
    if altitude is None:
        missing.append("orbit.altitude_km")
    elif not 150 <= float(altitude) <= 40000:
        warnings.append(f"altitude {altitude} km is outside the range this screen covers")
    inclination = orbit.get("inclination_deg")
    if inclination is not None and not 0 <= float(inclination) <= 180:
        warnings.append(f"inclination {inclination} deg is out of range")

    for index, entry in enumerate(profile.get("frequencies") or []):
        for field in ("band", "center_mhz", "bandwidth_mhz", "direction"):
            if entry.get(field) in (None, ""):
                missing.append(f"frequencies[{index}].{field}")
        if entry.get("direction") not in (None, "uplink", "downlink"):
            warnings.append(f"frequencies[{index}].direction should be uplink or downlink")

    return {"valid": not missing, "missing_fields": missing, "warnings": warnings,
            "mission_name": profile.get("mission_name", ""), "profile": profile,
            "tool_version": 1}


@tool(args_schema=ChecklistArgs)
def requirements_checklist(activity_type: str,
                           jurisdictions: list[str] | None = None) -> dict[str, Any]:
    """Build the licensing checklist for an activity from the sector rule file.

    An activity with no settled framework is flagged rather than forced into one.
    """
    rules = load_licensing_rules()
    activity = rules["activities"].get(activity_type)
    if activity is None:
        return {"error": f"unknown activity '{activity_type}'",
                "known": sorted(rules["activities"]), "tool_version": 1}

    items = []
    for name in activity["frameworks"]:
        framework = rules["frameworks"][name]
        for text in framework["items"]:
            items.append({
                "framework": name, "authority": framework["authority"],
                "reference": framework["reference"], "item": text, "status": "not_assessed",
            })

    novel = bool(activity.get("novel"))
    return {
        "activity_type": activity_type,
        "jurisdictions": jurisdictions or ["US"],
        "items": items, "item_count": len(items),
        "frameworks": activity["frameworks"],
        "no_established_framework": novel,
        "counsel_note": rules["novel_activity_note"] if novel else "",
        "tool_version": 1,
    }


@tool(args_schema=AffectedOperatorsArgs)
def find_affected_operators(frequencies: list[dict[str, Any]],
                            orbit: dict[str, Any] | None = None) -> dict[str, Any]:
    """Find registry entries whose occupied bandwidth overlaps the proposed frequencies.

    Overlap is computed from the occupied bandwidth on each side of the centre
    frequency, not from the centre alone, which is how overlaps get missed.
    """
    import pandas as pd

    ensure_samples()
    registry_path = SAMPLE_DIR / "spectrum_registry.csv"
    if not registry_path.is_file():
        return {"error": "spectrum registry is unavailable", "tool_version": 1}

    registry = pd.read_csv(registry_path)
    orbit = orbit or {}
    altitude = float(orbit.get("altitude_km") or 0)

    matches: list[dict[str, Any]] = []
    for entry in frequencies:
        try:
            centre = float(entry["center_mhz"])
            width = float(entry["bandwidth_mhz"])
        except (KeyError, TypeError, ValueError):
            continue
        low, high = centre - width / 2, centre + width / 2

        for _, row in registry.iterrows():
            other_low = float(row["center_mhz"]) - float(row["bandwidth_mhz"]) / 2
            other_high = float(row["center_mhz"]) + float(row["bandwidth_mhz"]) / 2
            overlap = min(high, other_high) - max(low, other_low)
            if overlap <= 0:
                continue
            same_regime = abs(float(row["altitude_km"]) - altitude) < 250 if altitude else False
            matches.append({
                "party": str(row["operator"]), "satellite": str(row["satellite"]),
                "band": str(row["band"]),
                "proposed_mhz": f"{low:.1f}-{high:.1f}",
                "registered_mhz": f"{other_low:.1f}-{other_high:.1f}",
                "overlap_mhz": round(overlap, 2),
                "same_orbital_regime": same_regime,
                "direction": str(row["direction"]),
            })

    parties = sorted({m["party"] for m in matches})
    return {"matches": sorted(matches, key=lambda m: -m["overlap_mhz"]),
            "affected_parties": parties, "party_count": len(parties),
            "checked_bands": len(frequencies), "tool_version": 1}


@tool(args_schema=CoordinationLetterArgs)
def draft_coordination_letter(party: str, overlaps: list[dict[str, Any]] | None = None,
                              mission: dict[str, Any] | None = None) -> dict[str, Any]:
    """Draft a coordination letter to one affected operator.

    Always a draft, with placeholders in brackets for anything not supplied.
    """
    mission = mission or {}
    overlaps = overlaps or []
    name = mission.get("mission_name", "[MISSION NAME]")
    operator = mission.get("operator", "[OPERATOR]")
    orbit = mission.get("orbit", {}) or {}

    rows = "\n".join(
        f"  - {o.get('band', '[BAND]')}: proposed {o.get('proposed_mhz', '[RANGE]')} MHz "
        f"overlaps {o.get('satellite', '[SATELLITE]')} at "
        f"{o.get('registered_mhz', '[RANGE]')} MHz by {o.get('overlap_mhz', '[N]')} MHz"
        for o in overlaps
    ) or "  - [OVERLAP DETAILS]"

    body = f"""DRAFT — for review by the operator and regulatory counsel. Not for sending.

To: {party}
From: {operator}
Subject: Frequency coordination for {name}

We are preparing a filing for {name}, a mission planned for an orbit at
{orbit.get('altitude_km', '[ALTITUDE]')} km and {orbit.get('inclination_deg', '[INCLINATION]')}
degrees inclination. A registry check indicates our proposed assignments overlap yours:

{rows}

We would like to open technical discussion before the filing deadline so that any
necessary sharing arrangements can be agreed. We propose a call in the week of
[PROPOSED DATE] and can share our interference analysis in advance.

Our technical point of contact is [CONTACT NAME], [CONTACT EMAIL].

Regards,
[SIGNATORY NAME]
{operator}
"""
    return {"title": f"DRAFT coordination letter to {party}", "body": body,
            "party": party, "overlap_count": len(overlaps),
            "placeholders": re.findall(r"\[([A-Z ]+)\]", body), "tool_version": 1}


@tool(args_schema=TimelineEstimateArgs)
def estimate_timeline(checklist: dict[str, Any]) -> dict[str, Any]:
    """Indicative duration ranges per framework and the longest path.

    Ranges come from the rule file and are indicative only; they are not commitments
    and coordination in particular has no guaranteed duration.
    """
    rules = load_licensing_rules()
    frameworks = checklist.get("frameworks") or []
    rows = []
    for name in frameworks:
        framework = rules["frameworks"].get(name)
        if not framework:
            continue
        low, high = framework["typical_months"]
        rows.append({"framework": name, "authority": framework["authority"],
                     "months_low": low, "months_high": high})

    critical = max(rows, key=lambda r: r["months_high"], default=None)
    return {
        "per_framework": rows,
        "critical_path": critical["framework"] if critical else None,
        "total_months_range": (f"{max((r['months_low'] for r in rows), default=0)}-"
                               f"{max((r['months_high'] for r in rows), default=0)}"),
        "note": "Indicative ranges only. Coordination has no guaranteed duration.",
        "tool_version": 1,
    }

## 6. Prompt addenda and workflows

Sector guidance and the three workflow definitions.

In [ ]:
ADDENDA = {
    "coordinator": (
        "Frame recommendations as maneuver-planning proposals for the operator. Always surface "
        "the probability of collision trend, miss-distance components, covariance quality, and "
        "data age. Novel in-space activities have no established licensing framework: say that "
        "regulatory counsel is required rather than forcing them into an existing checklist."
    ),
    "analyst": (
        "Probability of collision is reported in scientific notation with two significant "
        "figures. State the thresholds you compared against and whose policy they are."
    ),
    "researcher": (
        "Regulatory statements must cite a knowledge source. Anything you cannot source is "
        "marked 'needs verification' rather than stated."
    ),
    "executor": "Report the fields a message is missing by name rather than inferring them.",
}


class ConjunctionInputs(BaseModel):
    cdm_text: str = Field(default="", description="CCSDS CDM series, or leave empty for a sample.")
    sample_name: str = Field(default="cdm_high_risk", description="Bundled CDM series.")
    primary_norad: int | None = Field(default=None, description="Primary catalogue number.")
    secondary_norad: int | None = Field(default=None, description="Secondary catalogue number.")
    tca_utc: str = Field(default="", description="Time of closest approach, if known.")
    hard_body_radius_m: float = Field(default=HARD_BODY_RADIUS_M)
    operator_policy: dict[str, float] = Field(
        default_factory=lambda: {"pc_red": PC_RED, "pc_yellow": PC_YELLOW},
        description="Operator thresholds; these are policy, not physical constants.",
    )


class AnomalyInputs(BaseModel):
    sample_name: str = Field(default="telemetry_rw_fault", description="Bundled telemetry.")
    telemetry_path: str = Field(default="", description="Uploaded CSV instead of a sample.")
    anomaly_description: str = Field(default="", description="What was noticed, in plain words.")
    anomaly_window_utc: str = Field(default="", description="Optional window of interest.")
    target_channel: str = Field(default="rw1_current_a", description="Channel to explain.")


class LicensingInputs(BaseModel):
    sample_name: str = Field(default="mission_eo_leo", description="Bundled mission profile.")
    mission_name: str = Field(default="", description="Overrides the sample when given.")
    activity_type: str = Field(default="earth_observation")
    jurisdictions: list[str] = Field(default_factory=lambda: ["US"])


CONJUNCTION_PLAN = Plan(
    objective="Assess a conjunction and prepare a go/no-go brief for the operator.",
    steps=[
        PlanStep(id="s1", agent="executor",
                 instruction="Parse the conjunction data messages and report the latest "
                             "miss distance, time of closest approach, and both covariances.",
                 tool_hints=["parse_cdm"], expected_output="parsed messages"),
        PlanStep(id="s2", agent="analyst",
                 instruction="Compute the probability of collision across the series and "
                             "assess covariance quality, including the dilution check.",
                 tool_hints=["pc_trend", "covariance_quality"], depends_on=["s1"],
                 expected_output="Pc trend and data-quality findings"),
        PlanStep(id="s3", agent="analyst",
                 instruction="Classify the conjunction against the operator's thresholds.",
                 tool_hints=["classify_conjunction"], depends_on=["s2"],
                 expected_output="a level with reasons"),
        PlanStep(id="s4", agent="researcher",
                 instruction="Find the conjunction assessment practices and threshold "
                             "conventions relevant to this decision.",
                 tool_hints=["search_knowledge"], expected_output="cited context"),
    ],
)

ANOMALY_PLAN = Plan(
    objective="Rank plausible causes of a spacecraft anomaly from telemetry.",
    steps=[
        PlanStep(id="s1", agent="executor",
                 instruction="Load the telemetry and describe its channels and span.",
                 tool_hints=["load_telemetry"], expected_output="dataset summary"),
        PlanStep(id="s2", agent="analyst",
                 instruction="Detect anomalous intervals and correlate the target channel "
                             "against the others, noting which lead and which follow.",
                 tool_hints=["detect_anomalies", "correlate_channels"], depends_on=["s1"],
                 expected_output="flagged channels and lagged correlations"),
        PlanStep(id="s3", agent="researcher",
                 instruction="Find past cases and failure notes matching these symptoms.",
                 tool_hints=["search_knowledge"], expected_output="similar cases"),
        PlanStep(id="s4", agent="analyst",
                 instruction="Rank failure modes against the observed symptoms and list the "
                             "diagnostic checks that would separate them.",
                 tool_hints=["rank_hypotheses"], depends_on=["s2", "s3"],
                 expected_output="ranked hypotheses"),
    ],
)

LICENSING_PLAN = Plan(
    objective="Assemble a licensing and coordination packet for review.",
    steps=[
        PlanStep(id="s1", agent="executor",
                 instruction="Validate the mission profile and list anything missing.",
                 tool_hints=["validate_mission_profile"], expected_output="validated profile"),
        PlanStep(id="s2", agent="executor",
                 instruction="Build the requirements checklist for this activity and find "
                             "operators whose registered bandwidth overlaps the proposal.",
                 tool_hints=["requirements_checklist", "find_affected_operators"],
                 depends_on=["s1"], expected_output="checklist and affected parties"),
        PlanStep(id="s3", agent="researcher",
                 instruction="Draft a coordination letter to the most affected operator.",
                 tool_hints=["draft_coordination_letter"], depends_on=["s2"],
                 expected_output="a draft letter"),
        PlanStep(id="s4", agent="analyst",
                 instruction="Estimate indicative durations per framework and the critical path.",
                 tool_hints=["estimate_timeline"], depends_on=["s2"],
                 expected_output="indicative timeline"),
    ],
)

# Demo-mode scripts. Arguments are fixed, which is why every tool accepts a sample
# name or falls back to the most recent dataset.
CONJUNCTION_SCRIPT = {
    "s1": [{"tool_calls": [{"name": "parse_cdm", "args": {"sample_name": "cdm_high_risk"}}]},
           "parsed"],
    "s2": [{"tool_calls": [
        {"name": "pc_trend", "args": {"sample_name": "cdm_high_risk"}},
        {"name": "covariance_quality", "args": {"sample_name": "cdm_high_risk"}}]}, "assessed"],
    "s3": [{"tool_calls": [
        {"name": "classify_conjunction", "args": {"sample_name": "cdm_high_risk"}}]},
        "classified"],
    "s4": [{"tool_calls": [{"name": "search_knowledge", "args": {
        "query": "probability of collision thresholds and covariance quality", "k": 4}}]},
        "context gathered"],
}

ANOMALY_SCRIPT = {
    "s1": [{"tool_calls": [{"name": "load_telemetry",
                            "args": {"sample_name": "telemetry_rw_fault"}}]}, "loaded"],
    "s2": [{"tool_calls": [
        {"name": "detect_anomalies", "args": {}},
        {"name": "correlate_channels", "args": {"target": "rw1_current_a"}}]}, "analysed"],
    "s3": [{"tool_calls": [{"name": "search_knowledge", "args": {
        "query": "reaction wheel current rise bearing degradation", "k": 4,
        "include_cases": True}}]}, "cases found"],
    "s4": [{"tool_calls": [{"name": "rank_hypotheses", "args": {
        "symptoms": ["rw_current_rise", "rw_temp_rise", "rw_speed_oscillation"]}}]}, "ranked"],
}

LICENSING_SCRIPT = {
    "s1": [{"tool_calls": [{"name": "validate_mission_profile",
                            "args": {"sample_name": "mission_eo_leo"}}]}, "validated"],
    "s2": [{"tool_calls": [
        {"name": "requirements_checklist", "args": {"activity_type": "earth_observation",
                                                    "jurisdictions": ["US"]}},
        {"name": "find_affected_operators", "args": {
            "frequencies": [{"band": "X", "center_mhz": 8200.0, "bandwidth_mhz": 300.0,
                             "direction": "downlink"}],
            "orbit": {"altitude_km": 610}}}]}, "checklist built"],
    "s3": [{"tool_calls": [{"name": "draft_coordination_letter", "args": {
        "party": "Northstar Imaging",
        "mission": {"mission_name": "Northlight-1", "operator": "Northlight Imaging (fictional)",
                    "orbit": {"altitude_km": 610, "inclination_deg": 97.8}}}}]}, "drafted"],
    "s4": [{"tool_calls": [{"name": "estimate_timeline", "args": {
        "checklist": {"frameworks": ["fcc_space_station", "itu_filing", "debris_mitigation",
                                     "remote_sensing"]}}}]}, "estimated"],
}

CONJUNCTION_WORKFLOW = WorkflowSpec(
    id="space.conjunction_triage",
    name="Conjunction triage",
    description="Assess a close approach and prepare a go/no-go brief for the operator.",
    input_schema=ConjunctionInputs,
    accepted_uploads=[".txt", ".cdm", ".json"],
    step_template="1. parse the messages  2. compute Pc, trend and covariance quality  "
                  "3. classify against policy  4. gather assessment context",
    default_plan=CONJUNCTION_PLAN,
    fake_script={"steps": CONJUNCTION_SCRIPT},
    level_vocab=["RED", "YELLOW", "GREEN", "INSUFFICIENT_DATA"],
    forbidden_phrases=[r"\bmaneuver (?:was|has been) (?:executed|uploaded|commanded)\b",
                       r"\bsafe to ignore\b"],
    sample_name="cdm_high_risk",
    example_request=("Assess the attached CDM series for our satellite and prepare a "
                     "go/no-go brief."),
)

ANOMALY_WORKFLOW = WorkflowSpec(
    id="space.anomaly_rca",
    name="Anomaly root-cause assistant",
    description="Rank plausible causes of a spacecraft anomaly from telemetry evidence.",
    input_schema=AnomalyInputs,
    accepted_uploads=[".csv", ".json"],
    step_template="1. load telemetry  2. detect and correlate  3. find similar cases  "
                  "4. rank failure modes",
    default_plan=ANOMALY_PLAN,
    fake_script={"steps": ANOMALY_SCRIPT},
    level_vocab=["HYPOTHESES_READY", "NEEDS_MORE_DATA"],
    forbidden_phrases=[r"\broot cause is confirmed\b", r"\bcommand the spacecraft\b"],
    sample_name="telemetry_rw_fault",
    example_request="Reaction wheel 1 current spiked at 14:20 UTC. What could explain it?",
)

LICENSING_WORKFLOW = WorkflowSpec(
    id="space.licensing_packet",
    name="Licensing and spectrum packet",
    description="Build a licensing checklist, find affected operators, and draft coordination.",
    input_schema=LicensingInputs,
    accepted_uploads=[".json", ".txt", ".pdf"],
    step_template="1. validate the profile  2. build the checklist and find overlaps  "
                  "3. draft coordination  4. estimate the timeline",
    default_plan=LICENSING_PLAN,
    fake_script={"steps": LICENSING_SCRIPT},
    level_vocab=["PACKET_READY_FOR_REVIEW", "GAPS_FOUND", "COUNSEL_REQUIRED"],
    forbidden_phrases=[r"\bfiled\b", r"\blicense granted\b",
                       r"\bcoordination (?:is )?complete\b"],
    sample_name="mission_eo_leo",
    example_request="Prepare the licensing and coordination packet for this LEO imaging mission.",
)

## 7. Sector pack

Tool allowlists per role, and registration.

In [ ]:
SECTOR_PACK = SectorPack(
    **sector_identity(SECTOR_ID),
    tools=[
        ToolSpec(tool=parse_cdm, roles=["executor"]),
        ToolSpec(tool=fetch_gp_elements, roles=["executor"]),
        ToolSpec(tool=compute_pc_2d, roles=["analyst"]),
        ToolSpec(tool=pc_trend, roles=["analyst"]),
        ToolSpec(tool=covariance_quality, roles=["analyst"]),
        ToolSpec(tool=classify_conjunction, roles=["analyst"]),
        ToolSpec(tool=load_telemetry, roles=["executor"]),
        ToolSpec(tool=build_timeline, roles=["executor"]),
        ToolSpec(tool=detect_anomalies, roles=["analyst"]),
        ToolSpec(tool=correlate_channels, roles=["analyst"]),
        ToolSpec(tool=rank_hypotheses, roles=["analyst"]),
        ToolSpec(tool=validate_mission_profile, roles=["executor"]),
        ToolSpec(tool=requirements_checklist, roles=["executor"]),
        ToolSpec(tool=find_affected_operators, roles=["executor"]),
        ToolSpec(tool=draft_coordination_letter, roles=["researcher"]),
        ToolSpec(tool=estimate_timeline, roles=["analyst"]),
    ],
    workflows=[CONJUNCTION_WORKFLOW, ANOMALY_WORKFLOW, LICENSING_WORKFLOW],
    addenda=ADDENDA,
    ensure_samples=ensure_samples,
)

register_sector(SECTOR_PACK)

## Demo

Examples only; this cell is dropped from the built module.

In [ ]:
# Run every workflow offline and show what the reviewer would see.
import asyncio

ensure_samples()
for workflow in SECTOR_PACK.workflows:
    print(f"--- {workflow.id} ---")
    run_id = asyncio.run(start_run("space", workflow.id, workflow.example_request, {}))
    view = asyncio.run(get_run(run_id))
    print(view.status, view.brief["recommendation_level"] if view.brief else "no brief")

## Build check

Confirms the notebook reached the generated module.

In [ ]:
def hello() -> str:
    """Return this module's name, so the build pipeline can be checked end to end."""
    return "automatron_space"